## Setup and Dependencies

In [ ]:
%pip install langchain openai tiktoken langchain-openai python-dotenv langgraph genson

## Core Implementation

### Imports

In [1]:

import csv
import json
import operator
import os
import pprint
import time
import uuid
from getpass import getpass

from typing import Annotated, Any, Dict, List, Literal, Optional, Sequence, Union

from dotenv import load_dotenv
from IPython.display import display, Image
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage

from langgraph.constants import END, START
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver


ModuleNotFoundError: No module named 'langgraph'

### Loading Environment variables

In [ ]:
import getpass
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI


def get_key():
    """Get OpenAI API key from .env file, environment, or user input."""
    # Clear any cached API key
    if 'OPENAI_API_KEY' in os.environ:
        del os.environ['OPENAI_API_KEY']

    # Load environment variables from .env file (force reload)
    load_dotenv(override=True)

    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        api_key = getpass.getpass('Enter your OpenAI API key: ')
        os.environ['OPENAI_API_KEY'] = api_key

    # Verify key format
    if api_key and not api_key.startswith('sk-'):
        print("⚠️  Warning: API key doesn't start with 'sk-'")

    print(f"✅ API key loaded (ends with: ...{api_key[-10:] if api_key else 'None'})")
    return api_key


# Set up API key
get_key()
print("API keys configured")

## Define Pydantic classes

In [ ]:
class Question(BaseModel):
    id: str
    prompt: str
    next: Optional[Dict[str, Any]] = Field(default_factory=dict)

    class Config:
        extra = "allow"  # Allow additional fields that might be in the JSON


class ModuleInfo(BaseModel):
    id: str
    name: str

    class Config:
        extra = "allow"


class ModuleLoader(BaseModel):
    questions: List[Question]
    module: ModuleInfo
    diagnostic_criteria: Optional[Dict[str, Any]] = Field(alias="diagnosticCriteria", default=None)

    # Computed field for easy question access
    _questions_dict: Dict[str, Question] = {}

    class Config:
        extra = "allow"
        validate_by_name = True  # Allows both snake_case and camelCase

    def __init__(self, **data):
        super().__init__(**data)
        # Build questions dictionary after initialization
        self._questions_dict = {q.id: q for q in self.questions}

    @classmethod
    def from_file(cls, file_path: str) -> "ModuleLoader":
        """Load module data from JSON file"""
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return cls(**data)

    def get_question(self, qid: str) -> Question:
        """Get question by ID"""
        if qid not in self._questions_dict:
            raise KeyError(f"{qid!r} not found")
        return self._questions_dict[qid]

    def show_branches(self, qid: str) -> None:
        """Display branching logic for a question"""
        question = self.get_question(qid)
        print(f"Branching for {qid}:")
        pprint.pp(question.next)

    def print_summary(self, num_questions: int = 2) -> None:
        """Print module summary and first few questions"""
        print(f"Loaded {len(self.questions)} questions for Module {self.module.id}: {self.module.name}\n")

        for i, q in enumerate(self.questions[:num_questions], start=1):
            print(f"{i}. {q.id}: {q.prompt}\n")


class ClassificationResult(BaseModel):
    """evaluator node use this schema for its output."""
    reasoning: str
    classification: Union[Literal["yes", "no", "need more clarification", "N/A"], int]

class DiagnosisResult(BaseModel):
    """Diagnosis node use this schema for its output."""
    reasoning: str
    classification: Literal["Yes", "No"]
    red_flags: str

# Usage example
PATH = "modules/working/module_a.json"

# Load the module
module_e = ModuleLoader.from_file(PATH)

# Print summary
module_e.print_summary()

# Show branches for first question
module_e.show_branches("A1a")

## State class for Langgraph's State graph

In [ ]:
def add_conversation_log(left: list[dict[str, Any]], right: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Reducer function to add new entries to conversation log"""
    return left + right

class InterviewState(BaseModel):
    "Schema of shared state between all nodes of the M.I.N.I. Agent graph"
    messages: Annotated[Sequence[BaseMessage], add_messages] = Field(default_factory=list)
    conversation_log: Annotated[list[dict[str, Any]], add_conversation_log] = Field(default_factory=list)
    module_loader: Any
    current_question_id: Optional[str]
    question_path: List[str] = Field(default_factory=list)
    evaluator_results: Annotated[List[Any], operator.add] = Field(default_factory=list)
    module_completed: bool = False
    ready_for_diagnosis: bool = False




## Instantiating the core LLM

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
# Initialize LLM with structured output

evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
structured_llm = evaluator_llm.with_structured_output(ClassificationResult)
diagnosis_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1).with_structured_output(DiagnosisResult)


### questioner System Messasge Template

In [ ]:
SYSTEM_MESSAGE_TEMPLATE = """You are an AI agent with the role of a skilled clinician trained in professional psychological diagnosis and care conducting interview based on provided psychiatric interview questionnaire.

Current question to present:
{question}

Instructions:
- Present ONLY the above question as it is. DONT ASK ANY OTHER QUESTION.
- Keep your initial question presentation concise and clear
- Maintain an empathetic, professional tone throughout
- If the patient asks for clarification, provide brief, helpful explanations
- Do not interpret, evaluator, or respond to the content of patient answers
- Do not provide clinical advice, diagnosis, or therapeutic responses
- Stay focused on your role as the question presenter
- ask the patient to answer with yes or no. or if uncertain, tell 'feel free to explain' """


## questioner LLM Node

In [ ]:
def questioner_node(state: InterviewState):
    """questioner node that presents questions, then collects user reply."""
    print("enter questioner")

    # 1. Build the system prompt + history
    q = state.module_loader.get_question(state.current_question_id)
    sys_msg = SystemMessage(content=SYSTEM_MESSAGE_TEMPLATE.format(question=q))
    msgs = [sys_msg] + state.messages

    # 2. Invoke LLM to get the next question
    response = llm.invoke(msgs)
    print('\n\n' + 15* '--' + f" Start of question {state.current_question_id}" + 15* '--')
    print(response.content)  # show question to user

    # 3. Immediately read the user's answer
    user_input = input("You: ").strip()
    human_msg = HumanMessage(content=user_input)
    print(15* '--' + f" End of question {state.current_question_id}" + 15* '--' + '\n\n')
    print("exit questioner")
    # 4. Return both the LLM question and the new HumanMessage
    return {"messages": [response, human_msg]}


## evaluator Node

In [ ]:
def evaluator_node(state: InterviewState):
    # assume the loop-driver put the user’s message in state.messages[-1]
    print("enter evaluator")
    last_question = state.module_loader.get_question(state.current_question_id)
    patient_last_answer = state.messages[-1]
    prompt = f"""
You are a clinical psychological assessment agent. Analyze the patient's response to determine if it clearly answers the questioner's question.

CLASSIFICATION RULES: 
- "need more clarification" - Response is ambiguous, uncertain, irrelevant, incomplete, or confidence is ≤70%

ANALYSIS PROCESS:
1. Does the response directly address the question asked?
2. Is the patient's intent clear and unambiguous?
3. Are you >70% confident in the positive/negative classification?

If any answer above is "no", classify as "need more clarification".

EXAMPLES:
Question: "Do you often feel sad?"
- "Yes, almost every day" → yes
- "No, I'm generally happy" → no
- "Sometimes, I guess" → need more clarification
- "What do you mean by sad?" → need more clarification

Question: {last_question}
Patient Response: {patient_last_answer.content}

"""
    resp = structured_llm.invoke([HumanMessage(prompt)])

    # Access Pydantic model attributes directly
    print(f"reason behind the decision: {resp.reasoning}")
    print(f"decision: {resp.classification}")

    label = str(resp.classification)
    if label not in ("yes", "no", "need more clarification"):
        label = "need more clarification"

    print("exit evaluator")
    return {"messages": [AIMessage(str(resp.reasoning))], "evaluator_results": [str(resp.classification)]}

In [ ]:
def should_clarify(state: InterviewState) -> str:
    return "clarify" if state.evaluator_results[-1] == "need more clarification" else "continue"

In [ ]:
def create_conv_log(state: InterviewState) -> dict:
    return {
        "question_id": state.current_question_id,
        "patient_response": state.messages[-2].content,
        "classification": state.evaluator_results[-1],
        "reasoning": state.messages[-1].content
        }


## Node (question) selector

In [ ]:
def node_navigator_tool(state: InterviewState):
    """
    navigator node: reads the classification from the last message,
    then advances current_question_id or flags ready_for_diagnosis.
    """
    updates = {}
    print("enter node navigator")

    # Creating Conversation log instance before updating questions. why here? Because we enter this node only if
    # evaluator made yes/no decision; not 'need more clarification'
    updates["conversation_log"] = [create_conv_log(state)]
    
    # 1. Current question and its branches
    qid = state.current_question_id
    print(f"question_id: {qid}")
    branches = state.module_loader.get_question(qid).next
    print(f"branches: {branches}")

    # 2. Pull the evaluator’s classification from the last message
    evaluator_classification = state.evaluator_results[-1]
    print(f"evaluator_classification: {evaluator_classification}")
    if isinstance(evaluator_classification, int):
        evaluator_classification = "number"
    else:
        evaluator_classification = evaluator_classification.strip().lower()
    # print(f"raw evaluator label: {raw_label}")

    # 3. Normalize into one of our expected branch keys or default
    label = evaluator_classification if evaluator_classification in branches else "default"

    # 4. Pick the next question ID (or none)
    next_qid = branches.get(label)
    print(f"next question_id: {next_qid}")

    # 5. Update state
    if next_qid == "END_MODULE":
        updates["module_completed"] = True

    elif next_qid == "DIAGNOSTIC":
        updates["ready_for_diagnosis"] = True
        print("No further questions; ready for diagnosis")

    else:
        updates["current_question_id"] = next_qid
        state.question_path.append(next_qid)

    # print(updates)
    print("exit node navigator")
    return updates  #%% md

## Setup and Dependencies

In [ ]:
def should_diagnose(state: InterviewState) -> str:
    if state.ready_for_diagnosis:
        return "diagnose"
    if state.module_completed:
        return "end"
    else:
        return "continue"

## Diagnose Node

In [ ]:
from typing import Dict, Literal
from pydantic import BaseModel, ValidationError, RootModel

class CriteriaGroupResult(BaseModel):
    diagnosis: bool
    timeFrame: Literal['past', 'current', 'both', 'unknown']
    reasoning: str

class DiagnosisEvaluationOutput(RootModel[Dict[str, CriteriaGroupResult]]):
    pass

def diagnosis_node(state: InterviewState):
    """analyses the conversation log and gives a diagnosis based on the criteria"""
    print("enter diagnose node")
    criteria = state.module_loader.diagnostic_criteria

    module_name = state.module_loader.module.name


    patient_responses = {}
    for entry in state.conversation_log:
        qid = entry.get("question_id")
        classification = entry.get("classification")
        if qid and classification:
            patient_responses[qid] = classification.strip().upper()

    module_key = module_name.strip().replace(" ", "")
    module_key = module_key[0].lower() + module_key[1:]
    if module_key not in criteria:
        raise ValueError(f"No criteria found for module '{module_name}'.")

    criteria_groups = criteria[module_key]
    if not isinstance(criteria_groups, dict) or not criteria_groups:
        raise ValueError(f"No valid criteria groups found for module '{module_name}'.")

    summary = "\n".join(f"{qid}: {ans}" for qid, ans in sorted(patient_responses.items()))

    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
    result = {}

    for group_name, rule_list in criteria_groups.items():
        if isinstance(rule_list, list):
            if not rule_list:
                result[group_name] = {"diagnosis": False, "timeFrame": "unknown", "reasoning": "No criteria provided."}
                continue
            criteria_formatted = "\n".join(f"{i+1}. {crit}" for i, crit in enumerate(rule_list))
        elif isinstance(rule_list, str):
            criteria_formatted = rule_list.strip()
        else:
            result[group_name] = {"diagnosis": False, "timeFrame": "unknown", "reasoning": "Invalid criteria format."}
            continue

        prompt = (
            "You are a psychiatric diagnostic evaluator.\n\n"

            "=== Patient Summary ===\n"
            "The patient's answers to yes/no questions are listed below:\n"
            f"{summary}\n\n"

            f"=== Diagnostic Group: {group_name} ===\n"
            "Criteria to evaluate:\n"
            f"{criteria_formatted}\n\n"

            "=== Instructions ===\n"
            "You must assess whether each diagnostic criterion is met, based solely on the patient's answers.\n\n"

            "1. General Rules:\n"
            "   - Analyze each criterion independently.\n"
            "   - Provide a concise explanation for whether each is met.\n\n"

            "2. Symptom Group Criteria (e.g., 'at least 5 symptom groups'):\n"
            "   - Treat all questions that belong to the same symptom group as one unit.\n"
            "   - If **any** question in a group is marked 'YES' (whether past or current), consider that symptom group as positive.\n"
            "   - To satisfy the criterion, the required number of positive symptom groups must all come from **only one** time frame — either 'current' or 'past', not a mix.\n"
            "   - List which symptom groups are positive and under which time frame(s).\n\n"

            "3. Non-Time-Frame-Specific Items (e.g., A1a, A1b, A2):\n"
            "   - For items that do **not** include a time frame suffix (like _past or _current), evaluate them as TRUE or FALSE regardless of time frame.\n"
            "   - These items are valid if marked 'YES' in any context.\n\n"

            "4. Organic Cause / Rule-Out Criteria:\n"
            "   - For now, infer that organic causes are ruled out.\n\n"

            "5. Diagnosis Summary:\n"
            "   - After assessing all criteria, provide a final diagnosis summary.\n"
            "   - If **all** criteria are met, the diagnosis is 'Yes'. Otherwise, it is 'No'.\n"
            "   - For a positive diagnosis, also determine the **time frame**:\n"
            "     - Choose from: **past**, **current**, **both**, or **unknown**.\n"
            "     - This refers to the time frame in which the symptom group criteria were satisfied.\n"
            "     - Do not use 'both' unless both past and current **independently** meet the symptom threshold.\n\n"

            "=== Output Format ===\n"
            "Return ONLY a JSON object with these fields:\n"
            "{\n"
            '  "diagnosis": true/false,\n'
            '  "timeFrame": "past" | "current" | "both" | "unknown",\n'
            '  "reasoning": "<a concise explanation for the assessment>"\n'
            "}\n"
            "DO NOT include any other text outside this JSON.\n"
        )

        response = llm.invoke([HumanMessage(content=prompt)])
        content = response.content.strip()

        print(f"LLM JSON output for group '{group_name}':\n{content}\n{'-'*40}")

        try:
            data = json.loads(content)
            group_result = CriteriaGroupResult.model_validate(data)
        except (json.JSONDecodeError, ValidationError) as e:
            raise ValueError(f"Failed to parse LLM JSON output for group '{group_name}': {e}\nRaw output:\n{content}")

        result[group_name] = group_result.model_dump()

    return DiagnosisEvaluationOutput.model_validate(result)

## building Graph

In [ ]:
def create_questioner(module_path: str):
    """
    Build and compile the questioner workflow,
    ensuring the very first node is 'questioner',
    and render a Mermaid diagram of the state graph.
    """
    # Load your module
    module_loader = ModuleLoader.from_file(module_path)

    # Initialize memory & graph
    memory   = MemorySaver()
    workflow = StateGraph(InterviewState)

    # Register nodes
    workflow.add_node("questioner", questioner_node)
    workflow.add_node("evaluator",       evaluator_node)
    workflow.add_node("navigator",    node_navigator_tool)
    workflow.add_node("diagnosis",   diagnosis_node)

    # Wire START → questioner
    workflow.add_edge(START, "questioner")

    # After questioner, always evaluator
    workflow.add_edge("questioner", "evaluator")

    # Conditional routing after evaluator
    workflow.add_conditional_edges(
        "evaluator",
        should_clarify,
        {
            "clarify":  "questioner",
            "continue": "navigator",
        }
    )

    # Conditional routing after navigator
    workflow.add_conditional_edges(
        "navigator",
        should_diagnose,
        {
            "continue": "questioner",
            "diagnose":  "diagnosis",
            "end":       END,
        }
    )

    # Final diagnosis → end
    workflow.add_edge("diagnosis", END)

    # Compile the app with memory
    app = workflow.compile(checkpointer=memory)

    # Render the graph as a Mermaid PNG
    display(
        Image(
            app.get_graph(xray=True)
               .draw_mermaid_png()
        )
    )

    return app, module_loader


## Run Function

In [ ]:

def run_interview_session(module_path: str, starting_qid: str):
    """
    Run an interactive interview session, wired up to the questioner graph.
    """
    # 1. Compile the graph and load the module
    app, module_loader = create_mini_questioner(module_path)

    # 2. Ask for or create a thread ID

    thread_id = f"session_{uuid.uuid4().hex[:8]}"
    print(f"Created new session: {thread_id}")

    config = {
        "configurable": {"thread_id": thread_id},
        "recursion_limit": 100
    }

    # 3. Initialize the shared InterviewState
    state = InterviewState(
        messages=[],
        conversation_log=[],
        module_loader=module_loader,
        current_question_id=starting_qid,
        question_path=[],
        evaluator_results=[],
        module_completed=False,
        ready_for_diagnosis=False,
    )


    print(f"\nStarting interview session: {thread_id}\n")
    print("-" * 50)

    # 4. Main loop: read user input, inject into state, and invoke graph
    try:
       

        # 4a. Append the user's message to state.messages
        state.messages = list(state.messages) + [HumanMessage(content='start the interview')]

        # 4b. Invoke the workflow
        print("-----INVOKING------")
        result = app.invoke(state, config=config)
        print("-----END OF INVOKING-----")

        # 4.c  writing conversation log into report dictionary
        print("-----SAVING CONVERSATION LOG------")
        fieldnames = result["conversation_log"][0].keys()

        reports_dir = "reports"
        if not os.path.exists(reports_dir):
            os.makedirs(reports_dir)

        filename = f"mini_interview_{thread_id}.csv"
        save_path = os.path.join(reports_dir, filename)

        with open(save_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(result["conversation_log"])
        print(f"Conversation log saved at {save_path} ")


    except KeyboardInterrupt:
        state.session_metadata["end_time"] = time.time()
        print(f"\n\nInterrupted. You can resume later with session ID: {thread_id}")
    
    return result['conversation_log']

In [ ]:
result = run_interview_session(module_path="./modules/working/module_a.json".replace("./", os.getcwd() + os.sep, 1), starting_qid='A1a')